#### 1. Librerías.

In [16]:
%run "./librerias/librerias.ipynb"

#### 2. Constantes.

In [17]:
%run "./constantes/constantes.ipynb"

#### 3. Funciones.

In [18]:
%run "./funciones/funciones.ipynb"

#### 4. Conexión con la BBDD.

In [19]:
#a. Conexión a la BBDD.
conn = sqlite3.connect(path_db)

#### 5. Split Train y Test.

In [20]:
#a. Me traigo los id_lectores con 20 libros leídos, o más.
#id_lectores = pd.read_sql("""
#    SELECT DISTINCT id_lector
#      FROM interacciones
#    GROUP BY id_lector
#    HAVING COUNT(*) >= 20
#""", conn)["id_lector"]


#a. Me traigo todos los id_lectores (sin filtrar por cantidad de libros leídos).
# Los que tienen menos de 20 no se pueden evaluar, pero sirven como informativos en train.
id_lectores = pd.read_sql("""
    SELECT DISTINCT id_lector
      FROM interacciones
""", conn)["id_lector"]

In [21]:
#b. Creo el dataframe df_train y df_test con ID_Lector, ID_Libro, Rating (como interacciones).
#i. Creo los dataframes vacíos con las columnas de interés.
#df_train = pd.DataFrame({"id_lector": [], "id_libro": [], "rating": []})
#df_test = pd.DataFrame({"id_lector": [], "id_libro": [], "rating": []})

#ii. Recorro cada ID_Lector, sus libros leídos, y el rating con el que los evaluó.
# Los últimos 20 temporalmente, los guardo en df_test. Los restantes, en df_train.
# Voy concatenando cada lector, hasta tener la base completa.
#for id_lector in id_lectores:
#    id_libros = pd.read_sql("""
#        SELECT id_lector, id_libro, fecha, rating
#          FROM interacciones
#         WHERE id_lector = ?
#         ORDER BY fecha
#    """, conn, params=[id_lector])


#    df_train = pd.concat([df_train, id_libros[:-20]], axis=0)
#    df_test = pd.concat([df_test, id_libros[-20:]], axis=0)

#b. Creo el dataframe df_train y df_test con ID_Lector, ID_Libro, Rating (como interacciones).
#i. Creo las listas donde voy acumulando los pedazos de cada lector.
# Uso listas y concateno una sola vez al final: concatenar dentro del loop es cuadrático,
# y arrancar de un DataFrame vacío con dtypes float me rompe los tipos de id_lector/id_libro.
partes_train = []
partes_test = []

#ii. Recorro cada ID_Lector, sus libros leídos, y el rating con el que los evaluó.
# Los últimos 20 temporalmente, los guardo en df_test. Los restantes, en df_train.
# Si tiene menos de 20, va entero a train (no se puede armar un test de 20 para él).
for id_lector in id_lectores:
    id_libros = pd.read_sql("""
        SELECT id_lector, id_libro, fecha, rating
          FROM interacciones
         WHERE id_lector = ?
           AND fecha GLOB '[0-9][0-9]-[0-9][0-9]-[0-9][0-9][0-9][0-9]'
         ORDER BY substr(fecha, 7, 4) || substr(fecha, 4, 2) || substr(fecha, 1, 2)
    """, conn, params=[id_lector])

    if len(id_libros) < 20:
        partes_train.append(id_libros)
    else:
        partes_train.append(id_libros[:-20])
        partes_test.append(id_libros[-20:])

#iii. Concateno todo de una.
df_train = pd.concat(partes_train, ignore_index=True)
df_test = pd.concat(partes_test, ignore_index=True)

In [22]:
#c. Verificación.
#i. Generales.
print("Train:", df_train.shape, "| Test:", df_test.shape)
print("Lectores en train:", df_train["id_lector"].nunique())
print("Lectores en test:", df_test["id_lector"].nunique())
print("Test por lector (debería ser 20 siempre):", df_test.groupby("id_lector").size().unique())
print("Total filas:", len(df_train) + len(df_test), "| esperado: 461407")
print("Dtypes:", df_train.dtypes.to_dict())
print("\n")
#ii. Orden temporal del split.
f_train = pd.to_datetime(df_train["fecha"], format="%d-%m-%Y")
f_test = pd.to_datetime(df_test["fecha"], format="%d-%m-%Y")

#1. Rango global de cada conjunto.
print("Train:", f_train.min().date(), "→", f_train.max().date())
print("Test: ", f_test.min().date(), "→", f_test.max().date())

#2. Lo que importa: por lector, ¿todo test es posterior a todo train?
max_train = f_train.groupby(df_train["id_lector"]).max()
min_test = f_test.groupby(df_test["id_lector"]).min()

comp = pd.concat([max_train, min_test], axis=1, keys=["max_train", "min_test"]).dropna()
violaciones = (comp["min_test"] < comp["max_train"]).sum()

print("Lectores evaluables:", len(comp))
print("Lectores con solapamiento train/test:", violaciones)

#3. Inspección de un lector cualquiera.
lector = df_test["id_lector"].iloc[0]
print("TRAIN (últimas 5):")
print(df_train[df_train["id_lector"] == lector].tail(5)[["fecha", "id_libro"]])
print("\nTEST (primeras 5):")
print(df_test[df_test["id_lector"] == lector].head(5)[["fecha", "id_libro"]])

Train: (383547, 4) | Test: (77860, 4)
Lectores en train: 10585
Lectores en test: 3893
Test por lector (debería ser 20 siempre): [20]
Total filas: 461407 | esperado: 461407
Dtypes: {'id_lector': dtype('O'), 'id_libro': dtype('O'), 'fecha': dtype('O'), 'rating': dtype('int64')}


Train: 2008-02-24 → 2024-12-31
Test:  2008-02-29 → 2024-12-31
Lectores evaluables: 3805
Lectores con solapamiento train/test: 0
TRAIN (últimas 5):
          fecha                       id_libro
189  03-06-2024                 carta-al-padre
190  10-06-2024                      un-amor-3
191  24-06-2024  los-hermosos-anos-del-castigo
192  02-07-2024                    hadji-murat
193  02-07-2024        la-muerte-de-ivan-ilich

TEST (primeras 5):
        fecha                                    id_libro
0  04-07-2024                          rabos-de-lagartija
1  05-07-2024                       siete-cuentos-morales
2  19-07-2024                          asterix-en-bretana
3  26-07-2024  un-grito-de-amor-desde-el

In [23]:
#d. Cierro la conexión.
conn.close()

#### 6. Exportación.

In [24]:
#a. Exporto df_train.
df_train.to_csv(path_train_crudo, index=False)

In [25]:
#b. Exporto df_test.
df_test.to_csv(path_test_crudo, index=False)